# Task 3 — Sound Data Collection and Processing

**Goal:** turn each team member's voice recordings into clean, augmented, machine-learning-ready features
for the *voiceprint verification* model.

| Step | What we do | Output |
|---|---|---|
| 1 | Load recordings (mixed `.ogg` / `.aac` / `.mp4`), standardise to 16 kHz mono WAV | `data/audio_wav/` |
| 2 | Visualise every sample as a **waveform** and **spectrogram** and interpret them | `audio_outputs/*.png` |
| 3 | Apply **5 augmentations** (pitch up/down, time stretch fast/slow, background noise) | `data/audio_augmented/` |
| 4 | Extract **MFCCs, spectral roll-off, energy** (+ a few extras) | **`audio_features.csv`** |
| 5 | Validate the feature table | checks + PCA sanity plot |

The reusable functions live in `audio_features.py` so the command-line demo (Task 6) can use exactly the
same preprocessing on a live voice sample.

**Phrases recorded** (2 per member): *"Yes, approve"* and *"Confirm transaction"*.

## 1. Setup

In [ ]:
# pip install librosa soundfile pandas matplotlib scikit-learn   (ffmpeg must also be installed)
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import soundfile as sf
from IPython.display import Audio, display

import audio_features as af

AUDIO_DIR = Path("data/audio")            # raw recordings from the Drive folder
WAV_DIR   = Path("data/audio_wav")        # standardised copies
AUG_DIR   = Path("data/audio_augmented")  # augmented copies
PLOT_DIR  = Path("audio_outputs")         # saved figures
for d in (AUDIO_DIR, WAV_DIR, AUG_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

### Getting the recordings
Download the shared Drive folder into `data/audio/` (keep the `Name 1.ext` / `Name 2.ext` file names).
The cell below does it automatically if `gdown` is installed; skip it if the files are already there.

In [ ]:
if not any(AUDIO_DIR.iterdir()):
    import gdown   # pip install gdown
    gdown.download_folder(
        "https://drive.google.com/drive/folders/128l3FZ54lsOrfk_gyMe3p2VUXTy8rGJm",
        output=str(AUDIO_DIR), quiet=False)
print(sorted(p.name for p in af.list_audio_files(AUDIO_DIR)))

## 2. Load, standardise and inventory the recordings

Recordings came from different phones/apps, so formats differ (`.ogg`, `.aac`, `.mp4`). Each file is
converted to **mono, 16 kHz WAV**, leading/trailing silence is trimmed (`top_db=30`) and the peak level is
normalised, so differences between speakers reflect the *voice* rather than recording volume or format.

In [ ]:
records = []
clean = {}                                  # source file name -> cleaned waveform
for src in af.list_audio_files(AUDIO_DIR):
    member, n = af.parse_filename(src)
    wav = af.convert_to_wav(src, WAV_DIR)
    raw_y, _ = librosa.load(wav, sr=af.SR)
    y = af.load_clean(wav)
    clean[src.name] = y
    records.append({
        "member": member, "sample_id": n, "phrase": af.PHRASE_MAP.get(n, f"phrase_{n}"),
        "file": src.name, "format": src.suffix.lstrip("."),
        "raw_duration_s": round(len(raw_y) / af.SR, 2),
        "clean_duration_s": round(len(y) / af.SR, 2),
        "rms": round(float(np.sqrt(np.mean(y**2))), 4),
    })
inventory = pd.DataFrame(records)
display(inventory)
print(f"{len(inventory)} recordings from {inventory.member.nunique()} members "
      f"({inventory.groupby('member').size().min()}+ each)")

### Listen to the cleaned samples

In [ ]:
for _, r in inventory.iterrows():
    print(f"{r.member} — sample {r.sample_id} ({r.phrase})")
    display(Audio(clean[r.file], rate=af.SR))

## 3. Waveforms and spectrograms

* **Waveform** (amplitude vs. time): shows loudness, how long the speaker actually talks, pauses between words
  and clipping/noise.
* **Mel spectrogram** (frequency vs. time, colour = energy in dB): shows *which frequencies* carry energy.
  Horizontal bands are the speaker's harmonics (pitch and formants), which is what makes a voice distinctive.

In [ ]:
def plot_sample(ax_wave, ax_spec, y, title):
    librosa.display.waveshow(y, sr=af.SR, ax=ax_wave, color="tab:blue")
    ax_wave.set(title=f"{title} — waveform", xlabel="", ylabel="Amplitude")
    S = librosa.feature.melspectrogram(y=y, sr=af.SR, n_fft=af.N_FFT, hop_length=af.HOP, n_mels=64)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_db, sr=af.SR, hop_length=af.HOP, x_axis="time", y_axis="mel",
                                   ax=ax_spec, cmap="magma")
    ax_spec.set(title=f"{title} — mel spectrogram", ylabel="Hz")
    return img

for member, grp in inventory.groupby("member"):
    fig, axes = plt.subplots(2, len(grp), figsize=(6 * len(grp), 6), squeeze=False)
    for col, (_, r) in enumerate(grp.iterrows()):
        img = plot_sample(axes[0, col], axes[1, col], clean[r.file], f"{member} #{r.sample_id}: {r.phrase}")
    fig.colorbar(img, ax=axes[1, :], format="%+2.0f dB", shrink=0.8)
    fig.suptitle(f"{member}", fontsize=14, y=1.01)
    fig.subplots_adjust(hspace=0.5, wspace=0.25)
    fig.savefig(PLOT_DIR / f"{member.lower()}_waveform_spectrogram.png", bbox_inches="tight")
    plt.show()

### Quantitative summary to support the interpretation

In [ ]:
summary = []
for _, r in inventory.iterrows():
    y = clean[r.file]
    S = np.abs(librosa.stft(y, n_fft=af.N_FFT, hop_length=af.HOP))
    freqs = librosa.fft_frequencies(sr=af.SR, n_fft=af.N_FFT)
    mean_spec = S.mean(axis=1)
    f0 = librosa.yin(y, fmin=70, fmax=400, sr=af.SR, frame_length=2048)   # rough pitch track
    summary.append({
        "member": r.member, "sample": r.sample_id, "phrase": r.phrase,
        "duration_s": r.clean_duration_s,
        "median_pitch_Hz": round(float(np.median(f0)), 1),
        "spectral_centroid_Hz": round(float(librosa.feature.spectral_centroid(y=y, sr=af.SR, n_fft=af.N_FFT, hop_length=af.HOP).mean()), 0),
        "dominant_freq_Hz": round(float(freqs[mean_spec.argmax()]), 0),
        "energy_below_1kHz_%": round(float(100 * mean_spec[freqs < 1000].sum() / mean_spec.sum()), 1),
        "energy_above_4kHz_%": round(float(100 * mean_spec[freqs > 4000].sum() / mean_spec.sum()), 1),
    })
display(pd.DataFrame(summary))

### Interpretation  *(complete after running — look at your own plots and the table above)*

Points to comment on for the report:

* **Recording quality:** Are the waveforms free of clipping (flat tops at ±1) and is the background between words
  quiet? Differences in format (`.ogg` / `.aac` / `.mp4`) mean some clips are heavily compressed, which removes
  detail above roughly 4–8 kHz; that shows up as an empty top band in the spectrogram.
* **Phrase structure:** *"Yes, approve"* has two short bursts (two words); *"Confirm transaction"* has more
  syllables and should show 3–5 bursts. Check the waveforms match.
* **Speaker differences:** compare `median_pitch_Hz` and `spectral_centroid_Hz` across members. Higher pitch =
  harmonics spaced further apart in the spectrogram. Same member, different phrase → similar pitch; that
  consistency is what the voiceprint model relies on.
* **Duration:** clips are only a few seconds, so each gives a small number of frames; this is why we use
  augmentation to enlarge the dataset.

## 4. Augmentation

Five augmentations simulate realistic variation in how the same person might speak or be recorded:

| Name | Parameter | Real-world scenario |
|---|---|---|
| `pitch_up` / `pitch_down` | ±2 semitones | speaker slightly higher/lower (tired, excited, different mic) |
| `time_stretch_fast` / `time_stretch_slow` | ×1.15 / ×0.85 | speaking faster or slower |
| `background_noise` | white noise at 20 dB SNR | recording in a noisy room |

Augmented copies keep the **same `source_file`** so that, when training, they can be kept in the same
train/test split as their original (otherwise the model is tested on near-duplicates of what it trained on).

In [ ]:
demo_file = inventory.iloc[0].file
demo_y = clean[demo_file]
names = list(af.AUGMENTATIONS)
fig, axes = plt.subplots(2, len(names), figsize=(4 * len(names), 6))
for i, name in enumerate(names):
    y_aug = af.augment(demo_y, name)
    librosa.display.waveshow(y_aug, sr=af.SR, ax=axes[0, i], color="tab:orange" if name != "original" else "tab:blue")
    axes[0, i].set(title=name, xlabel="")
    S_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y_aug, sr=af.SR, n_fft=af.N_FFT, hop_length=af.HOP, n_mels=64), ref=np.max)
    librosa.display.specshow(S_db, sr=af.SR, hop_length=af.HOP, x_axis="time", y_axis="mel", ax=axes[1, i], cmap="magma")
fig.suptitle(f"Augmentations of {demo_file}", y=1.02)
fig.tight_layout()
fig.savefig(PLOT_DIR / "augmentation_comparison.png", bbox_inches="tight")
plt.show()

for name in names:
    print(name); display(Audio(af.augment(demo_y, name), rate=af.SR))

**Reading the figure:** pitch shifts move the harmonic bands up/down while keeping the duration; time stretching
changes duration but keeps the pitch; noise adds a faint uniform haze across all frequencies in the spectrogram.

## 5. Feature extraction → `audio_features.csv`

For every original + augmented clip we compute (summarised as mean and standard deviation over time):

* **MFCCs 1–13** — compact description of the spectral envelope (vocal-tract shape), the standard voiceprint feature
* **Spectral roll-off** (85 %) — frequency below which 85 % of the energy lies
* **Spectral centroid, bandwidth, flatness**, **zero-crossing rate**
* **RMS energy** (mean/std) and **total energy**, plus clip **duration**

In [ ]:
audio_df, clips = af.build_audio_dataset(AUDIO_DIR, WAV_DIR, AUG_DIR)
audio_df.to_csv("audio_features.csv", index=False)
print("Saved audio_features.csv:", audio_df.shape)
display(audio_df.head(8))

## 6. Validation

In [ ]:
meta_cols = ["member", "sample_id", "phrase", "source_file", "augmentation", "is_augmented"]
feat_cols = [c for c in audio_df.columns if c not in meta_cols]

print("Rows per member x augmentation:")
display(pd.crosstab(audio_df.member, audio_df.augmentation))

print("Missing values:", int(audio_df[feat_cols].isna().sum().sum()))
print("Infinite values:", int(np.isinf(audio_df[feat_cols]).sum().sum()))
print("Duplicate rows:", int(audio_df.duplicated().sum()))
print("Feature columns:", len(feat_cols))
display(audio_df[feat_cols].describe().T[["mean", "std", "min", "max"]].head(12))

### Do the features separate speakers?
A quick PCA on the standardised features. If the voiceprint features are informative, each member's points
(original + augmented) should cluster together. With only 2 recordings per member this is a sanity check, not a
performance measure.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = StandardScaler().fit_transform(audio_df[feat_cols])
pcs = PCA(n_components=2, random_state=42).fit(X)
Z = pcs.transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
for member, idx in audio_df.groupby("member").groups.items():
    ax.scatter(Z[idx, 0], Z[idx, 1], label=member, s=45, alpha=0.8)
ax.set(xlabel=f"PC1 ({pcs.explained_variance_ratio_[0]:.0%})", ylabel=f"PC2 ({pcs.explained_variance_ratio_[1]:.0%})",
       title="Audio features by member (PCA)")
ax.legend(); fig.tight_layout()
fig.savefig(PLOT_DIR / "audio_features_pca.png", bbox_inches="tight")
plt.show()

## 7. Notes for the next tasks

* **Train/test split (Task 4):** split by `source_file`, not by row, so augmented copies of one recording never
  appear in both train and test (`GroupShuffleSplit` / `GroupKFold`). With only 2 recordings per member, keep one
  recording (and its augmentations) for testing, or use leave-one-recording-out.
* **Unauthorized attempt (Task 6):** record someone who is *not* in the team saying the same phrase and run it through
  `af.extract_features_from_file(path)` → the voiceprint model should reject it.
* **Live inference:** `af.extract_features_from_file()` applies the same cleaning/features as training, but no
  augmentation. Make sure its column order matches `feat_cols` before calling `model.predict`.